# Lookahead safety

A feature stamped at time `T` may only use information that existed before `T`. When it
doesn't, a backtest quietly trades on the future and looks better than any live system could.
This notebook shows the timing conventions, the individual mechanisms, and the general check
that every feature passes.

In [ ]:
import numpy as np
import pandas as pd

import nordic_equity_features as nef
from nordic_equity_features.bars import build_bars
from nordic_equity_features.lookahead import compare, lookahead_violations, perturb_after
from nordic_equity_features.schema import TZ, session_grid, snap_to_next_bar

data = nef.generate(n_days=10, seed=7)

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#eeeeee"


def style(fig, title, height=520):
    fig.update_layout(
        title=dict(text=title, font=dict(size=18, color=INK)),
        height=height, margin=dict(l=60, r=20, t=70, b=40),
        plot_bgcolor="white", paper_bgcolor="white", font=dict(color=MUTED),
        hovermode="x unified", legend=dict(orientation="h", y=1.06, x=0),
    )
    fig.update_xaxes(gridcolor=GRID, linecolor=GRID)
    fig.update_yaxes(gridcolor=GRID, linecolor=GRID, zerolinecolor="#d0d0d0")
    return fig

## The bar clock

A bar labelled `T` contains trades in `[T − 1 min, T)`. A trade at exactly 10:00:00 belongs to
the 10:01 bar, because the 10:00 bar was already complete when it happened.

In [ ]:
day = data.days[0]
trades = pd.DataFrame({
    "timestamp": pd.DatetimeIndex([f"{day.date()} 09:59:59.999", f"{day.date()} 10:00:00"]).tz_localize(TZ),
    "price": [100.0, 101.0], "quantity": [10.0, 10.0],
})
build_bars(trades, [day]).loc[f"{day.date()} 10:00":f"{day.date()} 10:01", ["close", "volume"]]

Events use the same clock: they become visible in the first bar labelled strictly after them.

In [ ]:
events = pd.Series(pd.DatetimeIndex([f"{day.date()} 07:30", f"{day.date()} 10:15:00", f"{day.date()} 10:15:00.5"]).tz_localize(TZ))
pd.DataFrame({"event": events, "first bar that sees it": snap_to_next_bar(events, session_grid([day])).to_numpy()})

## The mechanisms

| Situation | Rule |
|---|---|
| Thresholds such as "a block trade is a top-5% trade" | Estimated on the **previous 20 days**, never the current day |
| End-of-day values (market cap) | Day D uses **day D−1**, because D's close is not known during D |
| Daily estimators (MedRV) | Shifted one day for the same reason |
| News | **Next bar** after the timestamp; pre-market items land at 09:01 |
| Splits and dividends | Visible only **after the announcement time** |
| Split-adjusted history | Built **as of** a time, applying only splits that have happened |
| Rolling statistics | **Trailing** windows ending at the current bar |

`tests/test_mechanisms.py` pins each of these with a small hand-made example.

## The general check

For a cutoff `t`, keep everything that was knowable before `t` and replace everything from `t`
on with an unrelated synthetic market. A split and a dividend announced exactly at `t` and a news
item stamped at `t` are planted too, because the boundary is where off-by-one errors hide. Then
recompute. Any feature that changes at a bar labelled `≤ t` used the future.

In [ ]:
TZ = "Europe/Stockholm"
cutoffs = {
    "mid-session": pd.Timestamp(f"{data.days[6].date()} 11:37", tz=TZ),
    "first bar of a day": pd.Timestamp(f"{data.days[5].date()} 09:01", tz=TZ),
    "dividend announcement minute": data.dividends.loc[0, "announced"],
    "before the open": pd.Timestamp(f"{data.days[4].date()} 08:00", tz=TZ),
    "last bar of a day": pd.Timestamp(f"{data.days[7].date()} 17:30", tz=TZ),
}
rows = []
for name, cutoff in cutoffs.items():
    for symbol in data.instruments.index:
        v = lookahead_violations(data, symbol, cutoff)
        rows.append({"cutoff": name, "symbol": symbol, "features changed before the cutoff": len(v)})
pd.DataFrame(rows).pivot(index="cutoff", columns="symbol", values="features changed before the cutoff")

Zero everywhere. To show that the check has teeth, here is a block-share feature with the
classic mistake: the 95th percentile of **today's** trade sizes. The morning bars then depend on
trades that happen in the afternoon.

In [ ]:
def same_day_block_share(d, symbol):
    t = d.trades.loc[d.trades["symbol"] == symbol]
    day = t["timestamp"].dt.tz_convert(TZ).dt.normalize()
    theta = t.groupby(day)["quantity"].transform(lambda q: q.quantile(0.95))
    label = t["timestamp"].dt.floor("1min") + pd.Timedelta("1min")
    share = t["quantity"].where(t["quantity"] >= theta, 0.0).groupby(label).sum() / t["quantity"].groupby(label).sum()
    return share.reindex(session_grid(d.days)).rename("block_share")

cutoff = cutoffs["mid-session"]
perturbed = perturb_after(data, cutoff)
safe = nef.compute_features(data, "NORDA")["block_share"]
safe_p = nef.compute_features(perturbed, "NORDA")["block_share"]
leaky = same_day_block_share(data, "NORDA")
leaky_p = same_day_block_share(perturbed, "NORDA")

print("prior-window threshold:", compare(safe.loc[:cutoff].to_frame(), safe_p.loc[:cutoff].to_frame()).to_dict("records"))
print("same-day threshold:    ", compare(leaky.loc[:cutoff].to_frame(), leaky_p.loc[:cutoff].to_frame()).to_dict("records"))

In [ ]:
from nordic_equity_features.transactions import prior_window_threshold

def same_day_threshold(d, symbol):
    t = d.trades.loc[d.trades["symbol"] == symbol]
    return t.groupby(t["timestamp"].dt.tz_convert(TZ).dt.normalize())["quantity"].quantile(0.95)

def morning(series_by_day):
    bars = session_grid(data.days)
    per_bar = pd.Series(series_by_day.to_numpy(), index=pd.DatetimeIndex(series_by_day.index).tz_localize(None).normalize())
    values = per_bar.reindex(bars.tz_convert(TZ).tz_localize(None).normalize()).to_numpy()
    return pd.Series(values, index=bars).loc[cutoff.normalize() + pd.Timedelta("9h"):cutoff]

def threshold(d, how):
    t = d.trades.loc[d.trades["symbol"] == "NORDA"]
    return same_day_threshold(d, "NORDA") if how == "same-day" else prior_window_threshold(t, d.days)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.12,
                    subplot_titles=("Same-day 95th percentile", "Previous 20 days' 95th percentile"))
for row, how in enumerate(["same-day", "prior-window"], start=1):
    for d, name, color, dash in [(data, "original data", BLUE, "solid"), (perturbed, "future replaced", ORANGE, "dot")]:
        s = morning(threshold(d, how).astype(float))
        fig.add_trace(go.Scatter(x=s.index, y=s, name=name, line=dict(color=color, width=2, dash=dash),
                                 showlegend=row == 1), row=row, col=1)
fig.update_yaxes(title_text="block size (shares)")
style(fig, "Replacing the afternoon moves the same-day threshold the morning already used", height=600).show()

The same check runs in the test suite (`tests/test_lookahead.py`) for every feature, every
instrument and five boundary cutoffs, and as the last asset of the Dagster pipeline.